# Claims Adjuster Notes — Table Setup

This notebook creates and populates the **ClaimsAdjusterNotes** table in your Fabric Lakehouse.

This table is designed to be used as a second data source alongside the Insurance Claims semantic model in a **Fabric Data Agent** demo.

**Table:** `ClaimsAdjusterNotes`  
**Rows:** 30 notes across 15 claims  
**Key scenarios covered:**
- Denied claims with customer disputes and escalations
- In-Review claims with pending documentation
- Approved claims with standard closure notes
- Multi-note claims showing a realistic investigation trail

---
> ⚙️ **Before running:** Make sure this notebook is attached to the Lakehouse
> where you want the table created. Update `LAKEHOUSE_NAME` in Cell 2 if needed.

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Update this if you want to write to a specific lakehouse by name.
# If this notebook is already attached to a lakehouse, you can leave it as None
# and the default attached lakehouse will be used.

LAKEHOUSE_NAME = None   # e.g. 'InsuranceDemo'  — or leave as None
TABLE_NAME     = 'ClaimsAdjusterNotes'
WRITE_MODE     = 'overwrite'   # 'overwrite' drops and recreates | 'append' adds rows

print(f"Table target : {TABLE_NAME}")
print(f"Write mode   : {WRITE_MODE}")
print(f"Lakehouse    : {LAKEHOUSE_NAME or 'Default attached lakehouse'}")

In [ ]:
# ── SAMPLE DATA ───────────────────────────────────────────────────────────────
# 30 adjuster notes covering:
#   - Denied / out-of-window claims  (ClaimIDs 1031–1034)  ← demo centrepiece
#   - In-Review / Submitted claims   (ClaimIDs 1007, 1013, 1022, 1029)
#   - Standard approved claims       (ClaimIDs 1001–1009 sample)
#
# NoteCategory values:
#   Initial Review | Follow Up | Escalation | Closure | Documentation Request

notes_data = [

    # ── DENIED CLAIM 1031 ─ Brandon Patel / Policy 111 (Expired 2022-12-31)
    # ClaimDate: 2023-03-14  →  74 days after policy expired
    (1,  1031, 'Sandra Liu',  '2023-03-15', 'Initial Review',
     'Claim received for auto accident dated 2023-03-14. Policy 111 expired '
     '2022-12-31. Claim date is 74 days outside the policy window. '
     'Denial notice issued to customer.'),

    (2,  1031, 'Sandra Liu',  '2023-03-20', 'Follow Up',
     'Customer Brandon Patel called to dispute denial. States he believed '
     'policy auto-renewed. No renewal payment found in billing system. '
     'Advised customer to contact billing department.'),

    (3,  1031, 'Sandra Liu',  '2023-03-28', 'Escalation',
     'Escalated to senior adjuster per customer request. Customer submitted '
     'a payment receipt but date on receipt is 2023-01-15 — three weeks '
     'after policy lapse. Payment does not reinstate prior coverage. '
     'Denial upheld pending legal review.'),

    (4,  1031, 'Sandra Liu',  '2023-04-10', 'Closure',
     'Legal review complete. Denial upheld. Customer notified in writing. '
     'Claim closed. Customer has been offered a new policy at current rates.'),

    # ── DENIED CLAIM 1032 ─ Mei Chen / Policy 112 (Expired 2022-05-31)
    # ClaimDate: 2022-08-05  →  66 days after policy expired
    (5,  1032, 'Tom Reyes',   '2022-08-06', 'Initial Review',
     'Property damage claim received for incident dated 2022-08-05. '
     'Policy 112 expired 2022-05-31. Claim is 66 days past policy end date. '
     'No late renewal or grace period on file. Denial issued.'),

    (6,  1032, 'Tom Reyes',   '2022-08-18', 'Follow Up',
     'No customer response to denial letter. Second notice mailed. '
     'Claim amount of $12,000 will not be approved under expired policy terms.'),

    (7,  1032, 'Tom Reyes',   '2022-09-05', 'Closure',
     'Customer confirmed receipt of denial. No further dispute raised. '
     'Claim permanently closed.'),

    # ── DENIED CLAIM 1033 ─ Anthony Russo / Policy 113 (Expired 2023-02-28)
    # ClaimDate: 2023-06-20  →  112 days after policy expired
    (8,  1033, 'Maria Osei',  '2023-06-21', 'Initial Review',
     'Medical claim submitted for date of service 2023-06-20. Health policy '
     '113 expired 2023-02-28. Claim is 112 days outside coverage window. '
     'This is the largest time gap in our current denied claims queue. '
     'Denial notice sent.'),

    (9,  1033, 'Maria Osei',  '2023-07-03', 'Escalation',
     'Customer Anthony Russo disputes denial. Claims his doctor\'s office '
     'submitted the claim late and that services were covered. '
     'Requested itemized provider invoice. Escalated for supervisor review.'),

    (10, 1033, 'Maria Osei',  '2023-07-15', 'Documentation Request',
     'Awaiting itemized invoice from provider St. Mercy Medical Center. '
     'Customer has 30 days to provide documentation or claim will be closed.'),

    (11, 1033, 'Maria Osei',  '2023-08-10', 'Closure',
     'No documentation received within 30-day window. Denial upheld. '
     'Claim closed. Customer advised to pursue provider billing dispute '
     'directly with St. Mercy Medical Center.'),

    # ── DENIED CLAIM 1034 ─ James Harrington / Policy 101 (Begins 2023-01-01)
    # ClaimDate: 2022-11-30  →  32 days BEFORE policy began
    (12, 1034, 'Sandra Liu',  '2022-12-01', 'Initial Review',
     'Theft claim received for incident dated 2022-11-30. Policy 101 '
     'does not begin until 2023-01-01. Claim predates policy start by '
     '32 days. Coverage was not in force at the time of the incident. '
     'Denial issued.'),

    (13, 1034, 'Sandra Liu',  '2022-12-08', 'Follow Up',
     'Customer James Harrington contacted. He was under the impression '
     'that signing the policy agreement provided immediate coverage. '
     'Policy documents clearly state effective date of 2023-01-01. '
     'Denial upheld.'),

    (14, 1034, 'Sandra Liu',  '2022-12-15', 'Closure',
     'Customer accepted denial outcome after reviewing policy documents. '
     'No further escalation. Claim closed.'),

    # ── IN REVIEW CLAIM 1007 ─ Lauren Fitzgerald / Policy 106
    (15, 1007, 'Sandra Liu',  '2024-09-12', 'Initial Review',
     'Large property damage claim received for $22,000. Damage reported '
     'following severe storm. Initial inspection scheduled for 2024-09-20.'),

    (16, 1007, 'Sandra Liu',  '2024-09-22', 'Follow Up',
     'First inspection complete. Inspector noted structural roof damage '
     'consistent with storm event. Contractor estimate of $21,800 received. '
     'Requesting second independent appraisal before approving full amount.'),

    (17, 1007, 'Sandra Liu',  '2024-10-15', 'Documentation Request',
     'Second appraisal came in at $19,500. Discrepancy of $2,300 between '
     'estimates. Awaiting customer selection of approved contractor from '
     'preferred vendor list before final approval amount is set.'),

    # ── IN REVIEW CLAIM 1013 ─ Kevin Nguyen / Policy 115
    (18, 1013, 'James Park',  '2024-03-16', 'Initial Review',
     'Life benefit claim submitted for $25,000. Beneficiary documentation '
     'received. Standard 30-day review period initiated per policy terms.'),

    (19, 1013, 'James Park',  '2024-04-02', 'Documentation Request',
     'Additional documentation required: certified death certificate and '
     'completed beneficiary claim form W-9. Notified claimant by mail.'),

    (20, 1013, 'James Park',  '2024-04-18', 'Follow Up',
     'Certified death certificate received. Still awaiting W-9 form. '
     'Extended review window by 15 days pending receipt.'),

    # ── IN REVIEW CLAIM 1022 ─ Lauren Fitzgerald / Policy 106
    (21, 1022, 'Sandra Liu',  '2025-01-31', 'Initial Review',
     'Second property claim filed by Lauren Fitzgerald on policy 106. '
     'Water damage to basement reported. Claim amount $11,200. '
     'Flagged for review given prior open claim on same policy.'),

    (22, 1022, 'Sandra Liu',  '2025-02-10', 'Follow Up',
     'Inspection confirmed separate water damage event unrelated to '
     'prior storm claim 1007. Both claims proceeding independently. '
     'Plumber estimate of $10,800 received and under review.'),

    # ── SUBMITTED CLAIM 1029 ─ Nathan Caldwell / Policy 109
    (23, 1029, 'James Park',  '2025-02-04', 'Initial Review',
     'Theft claim submitted for $5,500. Police report attached. '
     'Claim entered into system and assigned for initial review. '
     'Customer notified of 5-7 business day response window.'),

    # ── APPROVED CLAIM 1001 ─ James Harrington / Policy 101
    (24, 1001, 'Sandra Liu',  '2024-02-11', 'Initial Review',
     'Auto accident claim for $8,500. Police report and repair estimate '
     'provided at time of filing. Liability confirmed. Approved for $8,000 '
     'after applying $500 deductible.'),

    (25, 1001, 'Sandra Liu',  '2024-02-20', 'Closure',
     'Payment of $8,000 issued to customer. Repair shop confirmed '
     'work complete. Claim closed.'),

    # ── APPROVED CLAIM 1003 ─ Sofia Delgado / Policy 102
    (26, 1003, 'Sandra Liu',  '2024-03-23', 'Initial Review',
     'Property damage claim for $15,000 following kitchen fire. '
     'Fire department report attached. Cause confirmed as accidental. '
     'Approved for $14,500 after deductible.'),

    (27, 1003, 'Sandra Liu',  '2024-04-01', 'Closure',
     'Contractor payment of $14,500 issued directly to restoration company '
     'per customer request. Claim closed.'),

    # ── APPROVED CLAIM 1009 ─ Aisha Thompson / Policy 108
    (28, 1009, 'Tom Reyes',   '2024-08-28', 'Initial Review',
     'Rear-end collision claim for $9,100. Dashcam footage provided by '
     'customer clearly establishes fault. Approved for $8,800 after '
     'deductible. No subrogation action required.'),

    (29, 1009, 'Tom Reyes',   '2024-09-04', 'Closure',
     'Payment of $8,800 processed. Repair verified complete by shop. '
     'Customer satisfaction survey sent. Claim closed.'),

    # ── APPROVED CLAIM 1010 ─ Nathan Caldwell / Policy 109
    (30, 1010, 'James Park',  '2024-02-15', 'Initial Review',
     'Major property damage claim for $31,000 following flood event. '
     'Full documentation package received at filing. FEMA disaster '
     'declaration on file for the county. Approved for $30,000 up to '
     'policy sub-limit for flood damage. Claim closed same day.')
]

print(f"Sample rows prepared: {len(notes_data)}")

In [ ]:
# ── BUILD DATAFRAME ───────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DateType
)
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField('NoteID',        IntegerType(), nullable=False),
    StructField('ClaimID',       IntegerType(), nullable=False),
    StructField('AdjusterName',  StringType(),  nullable=False),
    StructField('NoteDate',      StringType(),  nullable=False),  # cast to date below
    StructField('NoteCategory',  StringType(),  nullable=False),
    StructField('NoteText',      StringType(),  nullable=False),
])

df = spark.createDataFrame(notes_data, schema=schema)

# Cast NoteDate string → DateType
df = df.withColumn('NoteDate', to_date(col('NoteDate'), 'yyyy-MM-dd'))

print("Schema:")
df.printSchema()
print(f"Row count: {df.count()}")

In [ ]:
# ── PREVIEW DATA ─────────────────────────────────────────────────────────────
# Show the denied claim notes — these are the centrepiece of the demo

print("=== Denied Claim Notes (ClaimIDs 1031–1034) ===")
df.filter(col('ClaimID').isin([1031, 1032, 1033, 1034])) \
  .select('NoteID', 'ClaimID', 'AdjusterName', 'NoteDate', 'NoteCategory') \
  .orderBy('ClaimID', 'NoteDate') \
  .show(20, truncate=False)

print("\n=== Note Category Distribution ===")
df.groupBy('NoteCategory').count().orderBy('count', ascending=False).show()

print("\n=== Claims With Multiple Notes ===")
df.groupBy('ClaimID').count() \
  .filter(col('count') > 1) \
  .orderBy('count', ascending=False) \
  .show()

In [ ]:
# ── WRITE TO LAKEHOUSE ────────────────────────────────────────────────────────
# Writes as a Delta table — visible in the Lakehouse explorer and
# available to add as a data source in a Fabric Data Agent.

if LAKEHOUSE_NAME:
    write_path = f"Tables/{TABLE_NAME}"
    # Switch to the specified lakehouse context
    spark.sql(f"USE {LAKEHOUSE_NAME}")
else:
    write_path = f"Tables/{TABLE_NAME}"

df.write \
  .format('delta') \
  .mode(WRITE_MODE) \
  .option('overwriteSchema', 'true') \
  .saveAsTable(TABLE_NAME)

print(f"✅ Table '{TABLE_NAME}' written successfully ({WRITE_MODE} mode).")
print(f"   Rows written: {df.count()}")

In [ ]:
# ── VERIFY ───────────────────────────────────────────────────────────────────
# Read back from the Delta table to confirm it landed correctly

verify_df = spark.sql(f"SELECT * FROM {TABLE_NAME} ORDER BY ClaimID, NoteDate")

print(f"Rows in table: {verify_df.count()}")
print("\nFull table preview:")
verify_df.select('NoteID', 'ClaimID', 'AdjusterName', 'NoteDate', 'NoteCategory') \
         .show(30, truncate=False)

print("\n✅ Verification complete. Table is ready to add to your Fabric Data Agent.")
print("   In the Data Agent editor, click '+ Add data source' and")
print(f"   select this Lakehouse → Tables → {TABLE_NAME}")

---
## Next Steps

1. **Add to your Data Agent**  
   In the Fabric Data Agent editor → **+ Add data source** → select this Lakehouse → `ClaimsAdjusterNotes`

2. **Update your Agent Instructions**  
   Add this paragraph to the *Your Data* section of the agent prompt:
   > *`ClaimsAdjusterNotes` (Lakehouse table) — free-text operational notes written by adjusters during claim processing. Each note has a ClaimID, AdjusterName, NoteDate, NoteCategory (Initial Review, Follow Up, Escalation, Documentation Request, Closure), and NoteText. Use this table to provide narrative context when investigating specific claims.*

3. **Try these cross-source demo questions:**
   - *"Tell me everything about ClaimID 1031 — claim details and all adjuster notes"*
   - *"Show me all denied claims and any notes that mention the word dispute"*
   - *"Which claims in review have an escalation note?"*
   - *"What is the most recent adjuster note on the Brandon Patel claim?"*